# Predicting Student Exam Performance
### A Regression-Based Analysis of Behavioural, Familial, and School-Related Factors Affecting Academic Outcomes

**IDRA Capstone Project 2026**
**Name:** febin_thomas &nbsp;|&nbsp; **Institute:** Vimal Jyothi Engineering College, Chemperi
**Roll No.:** Vml23cs111 &nbsp;|&nbsp; **Enrollment No.:** IDRA-2026-569794

This notebook contains the complete, reproducible code for the capstone project. It runs top to bottom:
imports → data loading → cleaning → preprocessing/feature engineering → EDA → visualisation →
statistical analysis → model training → model evaluation → saving the cleaned dataset.

The accompanying PDF report (`febin_thomas_Capstone_2026.pdf`) tells the full analytical story with
interpretation; this notebook is the computational record behind it.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)

## 2. Data Loading and Inspection

The dataset (`P_2_StudentPerformanceFactors.csv`) was provided as one of the ten approved
capstone datasets in the IDRA LMS Study Material. We start by loading it and inspecting its
shape, column types, and completeness.

In [ ]:
df_raw = pd.read_csv("P_2_StudentPerformanceFactors.csv")
print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
# Missing values by column
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

In [ ]:
# Duplicate rows
print("Duplicate rows:", df_raw.duplicated().sum())

## 3. Data Cleaning

**Missing values:** Three categorical columns have missing values (all under 1.4% of rows), so we
impute with the column mode rather than dropping rows, to avoid unnecessary data loss or bias.

**Duplicates:** None were found, so no rows are removed on this basis.

**Outliers:** Numeric columns are checked with the IQR method. Real but extreme study/tutoring
behaviour is retained; only the impossible `Exam_Score` values above 100 are corrected (capped),
since a score above 100 is a data-entry artefact, not a legitimate observation.

In [ ]:
df = df_raw.drop_duplicates().copy()

for col in ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Outlier detection via IQR (numeric columns)
num_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_summary = {}
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[c] < lower) | (df[c] > upper)).sum()
    outlier_summary[c] = {"n_outliers": int(n_out), "lower": round(lower, 2), "upper": round(upper, 2)}

pd.DataFrame(outlier_summary).T

In [ ]:
# Exam_Score cannot exceed 100 -- cap any impossible values
impossible = (df["Exam_Score"] > 100).sum()
print("Exam_Score values above 100 (before fix):", impossible)
if impossible > 0:
    df["Exam_Score"] = df["Exam_Score"].clip(upper=100)
print("Exam_Score max after fix:", df["Exam_Score"].max())

In [ ]:
# Validation: before vs after
print("Rows before:", len(df_raw), "| Rows after:", len(df))
print("Missing before:", df_raw.isnull().sum().sum(), "| Missing after:", df.isnull().sum().sum())

## 4. Data Preprocessing and Feature Engineering

- **Ordinal encoding** for naturally ordered categories (Low/Medium/High).
- **One-hot encoding** for nominal categories (drop_first=True to avoid the dummy-variable trap).
- **Feature engineering:** `Engagement_Index`, `Study_Sleep_Ratio`, and `High_Attendance` are
  derived to capture combined behavioural effects not present in any single raw column.

In [ ]:
ordinal_map = {"Low": 1, "Medium": 2, "High": 3}
ordinal_cols = ["Parental_Involvement", "Access_to_Resources", "Motivation_Level",
                "Family_Income", "Teacher_Quality"]
for c in ordinal_cols:
    df[c + "_Ord"] = df[c].map(ordinal_map)

df["Engagement_Index"] = df["Hours_Studied"] + df["Attendance"] / 10 + df["Tutoring_Sessions"] * 2
df["Study_Sleep_Ratio"] = df["Hours_Studied"] / df["Sleep_Hours"].replace(0, np.nan)
df["Study_Sleep_Ratio"] = df["Study_Sleep_Ratio"].fillna(df["Study_Sleep_Ratio"].median())
df["High_Attendance"] = (df["Attendance"] >= df["Attendance"].median()).astype(int)

df[["Engagement_Index", "Study_Sleep_Ratio", "High_Attendance"]].describe()

In [ ]:
# Save the cleaned (pre-encoding) dataset -- this is the file submitted alongside the notebook
df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv with shape:", df.shape)

In [ ]:
# One-hot encode nominal (unordered) categorical columns for modelling
cat_cols = df.select_dtypes(include="object").columns.tolist()
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

model_df = df.drop(columns=ordinal_cols)
model_df = pd.get_dummies(model_df, columns=nominal_cols, drop_first=True)
print("Model-ready shape:", model_df.shape)
model_df.head()

## 5. Exploratory Data Analysis

We examine the correlation of each numeric predictor with the target, and compare `Exam_Score`
across key categorical groups, to identify which variables are worth carrying into the model.

In [ ]:
corr_target = df[num_cols].corr()["Exam_Score"].sort_values(ascending=False)
corr_target

In [ ]:
df.groupby("Parental_Involvement")["Exam_Score"].mean().reindex(["Low", "Medium", "High"])

In [ ]:
df.groupby("Peer_Influence")["Exam_Score"].mean().reindex(["Negative", "Neutral", "Positive"])

## 6. Statistical Analysis

Descriptive statistics (mean, median equivalent via 50%, std, IQR, skewness) for all numeric
variables. Skewness helps decide whether the mean or median better represents the "typical" value.

In [ ]:
desc = df[num_cols].describe().T
desc["skew"] = df[num_cols].skew()
desc.round(2)

## 7. Data Visualisation

Each chart below answers a specific question raised during EDA. Figures are numbered to match
the PDF report.

In [ ]:
# Figure 1: Distribution of Exam Score
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.histplot(df["Exam_Score"], kde=True, bins=30, color="#4C72B0", ax=ax)
ax.set_title("Distribution of Exam Score")
ax.set_xlabel("Exam Score"); ax.set_ylabel("Frequency")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 2: Parental Involvement vs Exam Score
fig, ax = plt.subplots(figsize=(7, 4.5))
order = ["Low", "Medium", "High"]
sns.boxplot(data=df, x="Parental_Involvement", y="Exam_Score", order=order, hue="Parental_Involvement",
            palette="Blues", legend=False, ax=ax)
ax.set_title("Parental Involvement Compared Across Exam Score")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 3: Peer Influence vs Exam Score
fig, ax = plt.subplots(figsize=(7, 4.5))
order = ["Negative", "Neutral", "Positive"]
sns.boxplot(data=df, x="Peer_Influence", y="Exam_Score", order=order, hue="Peer_Influence",
            palette="Greens", legend=False, ax=ax)
ax.set_title("Peer Influence Compared Across Exam Score")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 4: Hours Studied vs Exam Score
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.regplot(data=df, x="Hours_Studied", y="Exam_Score", scatter_kws={"alpha": 0.25, "s": 15},
            line_kws={"color": "red"}, ax=ax)
ax.set_title("Relationship Between Hours Studied and Exam Score")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 5: Attendance vs Exam Score
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.regplot(data=df, x="Attendance", y="Exam_Score", scatter_kws={"alpha": 0.25, "s": 15},
            line_kws={"color": "red"}, ax=ax)
ax.set_title("Relationship Between Attendance and Exam Score")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 6: Correlation heatmap
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Heatmap of Numerical Variables")
plt.tight_layout(); plt.show()

## 8. Machine Learning Methodology

**Problem type:** Regression (continuous target: `Exam_Score`).
**Split:** 80/20 train-test, `random_state=42` for reproducibility.
**Scaling:** `StandardScaler` fit on the training set only (no leakage), applied to Linear
Regression. Random Forest uses the unscaled features (tree-based models are scale-invariant).

In [ ]:
X = model_df.drop(columns=["Exam_Score"])
y = model_df["Exam_Score"]
feature_names = X.columns.tolist()
print("Number of features:", len(feature_names))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Train size:", len(X_train), "| Test size:", len(X_test))

In [ ]:
scaler = StandardScaler()
num_feature_cols = [c for c in feature_names if X[c].nunique() > 2]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_feature_cols] = scaler.fit_transform(X_train[num_feature_cols])
X_test_scaled[num_feature_cols] = scaler.transform(X_test[num_feature_cols])

In [ ]:
# Train Linear Regression (interpretable baseline)
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Train Random Forest (non-linear alternative)
rf_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Both models trained.")

## 9. Model Results and Evaluation

We evaluate both models on the held-out test set with MAE, MSE, RMSE, and R², then compare
train vs test performance to check for overfitting/underfitting.

In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": round(mae, 3), "MSE": round(mse, 3), "RMSE": round(rmse, 3), "R2": round(r2, 4)}

pred_lr_train = lr_model.predict(X_train_scaled)
pred_lr_test = lr_model.predict(X_test_scaled)
pred_rf_train = rf_model.predict(X_train)
pred_rf_test = rf_model.predict(X_test)

results = pd.DataFrame({
    "Linear Regression (Train)": regression_metrics(y_train, pred_lr_train),
    "Linear Regression (Test)":  regression_metrics(y_test, pred_lr_test),
    "Random Forest (Train)":     regression_metrics(y_train, pred_rf_train),
    "Random Forest (Test)":      regression_metrics(y_test, pred_rf_test),
}).T
results

In [ ]:
# Figure 7: Actual vs Predicted (Linear Regression, test set)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(y_test, pred_lr_test, alpha=0.3, s=18, color="#4C72B0")
lims = [min(y_test.min(), pred_lr_test.min()), max(y_test.max(), pred_lr_test.max())]
ax.plot(lims, lims, "r--", label="Perfect Prediction")
ax.set_xlabel("Actual Exam Score"); ax.set_ylabel("Predicted Exam Score")
ax.set_title("Actual vs Predicted Exam Score (Linear Regression)")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Figure 8: Residual distribution
resid = y_test - pred_lr_test
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.histplot(resid, kde=True, bins=30, color="#DD8452", ax=ax)
ax.axvline(0, color="black", linestyle="--")
ax.set_title("Distribution of Residuals (Linear Regression)")
ax.set_xlabel("Residual (Actual - Predicted)")
plt.tight_layout(); plt.show()

In [ ]:
# Feature influence: Linear Regression coefficients
coef = pd.Series(lr_model.coef_, index=feature_names).sort_values(key=abs, ascending=False)
coef.head(10).round(3)

In [ ]:
# Feature influence: Random Forest importances
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.head(10).round(4)

## 10. Summary

- Linear Regression outperforms Random Forest on the test set (higher R², lower RMSE), and shows
  a much smaller train/test gap -- evidence the underlying relationships are close to linear and
  that Linear Regression generalises better here.
- **Attendance** and the engineered **Engagement_Index** are the strongest predictors of
  `Exam_Score` in both models, consistent with the EDA correlation results.
- Full interpretation, limitations, and recommendations are presented in the accompanying PDF
  report, `febin_thomas_Capstone_2026.pdf`.